# Training NER (task 1) — Colab

Melatih baseline **BiLSTM-CRF** dan **IndoBERT-base-p1** di NERGrit (label PER, ORG, LOC), masing-masing 3 seed.
Semua logika ada di `src/indonesian_hoax_checker/ner/`; notebook ini hanya menjalankannya.

Cara pakai:
1. `Runtime > Change runtime type > GPU`.
2. Pastikan kode sudah di-push ke branch `BRANCH` di GitHub.
3. Jalankan semua sel dari atas. Hasil disalin ke Google Drive (`DRIVE_DIR`).

In [ ]:
# Pengaturan
REPO_URL = "https://github.com/danenftyessir/indonesian-hoax-checker.git"
BRANCH = "feat/ner"
DRIVE_DIR = "/content/drive/MyDrive/indonesian-hoax-checker"
SEEDS = "13 42 2026"

## 1. Cek GPU dan sambungkan Google Drive

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount("/content/drive")

## 2. Ambil kode dan install dependensi

Kalau repo private, simpan token GitHub di Colab Secrets dengan nama `GITHUB_TOKEN`.

In [ ]:
import os

# Pakai token kalau repo private
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    url = REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
%cd /content/repo
!git pull -q
!git log --oneline -1
!pip install -q -e ".[training]"

## 3. Siapkan data NERGrit dan NERP

In [ ]:
!python -m indonesian_hoax_checker.ner.prepare

## 4. Latih BiLSTM-CRF (baseline)

In [ ]:
!python -m indonesian_hoax_checker.ner.train --config configs/models/ner_bilstm_crf.yaml --seeds {SEEDS} 2>&1 | grep -v Warning

## 5. Latih IndoBERT-base-p1

In [ ]:
!python -m indonesian_hoax_checker.ner.train --config configs/models/ner_indobert.yaml --seeds {SEEDS} 2>&1 | grep -v Warning

## 6. Ringkasan hasil (F1 test, rata-rata ± simpangan baku 3 seed)

In [ ]:
import json
import pandas as pd

rows = []
for name in ["ner_bilstm_crf", "ner_indobert"]:
    path = f"outputs/metrics/{name}_summary.json"
    if not os.path.exists(path):
        continue
    s = json.load(open(path))
    row = {"model": name, "F1": f"{s['test_f1']['mean']:.4f} ± {s['test_f1']['std']:.4f}"}
    for kind, v in s["test_f1_per_type"].items():
        row[kind] = f"{v['mean']:.4f} ± {v['std']:.4f}"
    rows.append(row)
pd.DataFrame(rows)

## 7. Simpan hasil ke Google Drive

In [ ]:
# Checkpoint, metrik, dan prediksi hilang saat sesi Colab berakhir, jadi salin ke Drive
for folder in ["artifacts/checkpoints/ner", "outputs/metrics", "outputs/predictions"]:
    !mkdir -p "{DRIVE_DIR}/{folder}"
    !cp -r {folder}/. "{DRIVE_DIR}/{folder}/"
!ls -R "{DRIVE_DIR}" | head -40